In [5]:
!pip install ultralytics opencv-python-headless -q

In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
save_path = '/content/drive/MyDrive/yolo_detection'
os.makedirs(save_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # avtomatik yüklənir

results = model('/content/drive/MyDrive/highway_video.mp4', show=True)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/836) /content/drive/MyDrive/highway_video.mp4: 640x384 (no detections), 81.9ms
video 1/1 (frame 2/836) /content/drive/MyDrive/highway_video.mp4: 640x384 (no detections), 7.2ms
video 1/1 (frame 3/836) /content/drive/MyDrive/highway_video.mp4: 640x384 (no detections), 6.8ms
video 1/1 (frame 4/836) /content/drive/MyDrive/highway_video.mp4: 640x384 (no detections), 6.8ms
video 1/1 (frame 5/836) /content/drive/MyDrive/highway_video.mp4: 640

In [11]:
from ultralytics import YOLO
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

model = YOLO("yolov8n.pt")
VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

def analyze_congestion(frame):
    results = model(frame, verbose=False)[0]
    frame_area = frame.shape[0] * frame.shape[1]

    total_box_area = 0
    vehicles = []

    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        if cls_id in VEHICLE_CLASSES and conf > 0.3:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            box_area = (x2 - x1) * (y2 - y1)
            total_box_area += box_area
            vehicles.append({"class": VEHICLE_CLASSES[cls_id], "bbox": (x1,y1,x2,y2), "conf": conf})

            # Draw box
            color = (0, 255, 0) if cls_id == 2 else (255, 165, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"{VEHICLE_CLASSES[cls_id]} {conf:.0%}", (x1, y1-8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    coverage = total_box_area / frame_area

    if coverage > 0.45:
        status, color = "CONGESTED", (0, 0, 255)
    elif coverage > 0.25:
        status, color = "HEAVY", (0, 100, 255)
    elif coverage > 0.10:
        status, color = "NORMAL", (0, 200, 255)
    else:
        status, color = "FREE FLOW", (0, 200, 0)

    # Info panel
    cv2.rectangle(frame, (10, 10), (400, 110), (0, 0, 0), -1)
    cv2.putText(frame, f"Vehicles: {len(vehicles)}", (20, 40),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Coverage: {coverage:.1%}", (20, 70),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Status: {status}", (20, 100),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    return frame, len(vehicles), coverage, status

print("Ready! Use with video or image.")

Ready! Use with video or image.


In [12]:
cap = cv2.VideoCapture('/content/drive/MyDrive/highway_video.mp4')

vehicle_classes = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

frame_results = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Hər 30-cu kadrı analiz et (saniyədə 1 dəfə)
    if frame_count % 30 != 0:
        frame_count += 1
        continue

    results = model(frame, verbose=False)
    result = results[0]

    # Nəqliyyat say
    count = {'car': 0, 'motorcycle': 0, 'bus': 0, 'truck': 0}
    total_box_area = 0
    frame_area = frame.shape[0] * frame.shape[1]

    for box in result.boxes:
        cls_id = int(box.cls[0])
        if cls_id in vehicle_classes:
            count[vehicle_classes[cls_id]] += 1
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            total_box_area += (x2 - x1) * (y2 - y1)

    coverage = total_box_area / frame_area
    total = sum(count.values())

    if coverage > 0.45:
        status = "Tıxac"
    elif coverage > 0.25:
        status = "Sıx"
    elif coverage > 0.10:
        status = "Normal"
    else:
        status = "Boş"

    frame_results.append({
        'frame': frame_count,
        'second': frame_count // 30,
        'total_vehicles': total,
        'breakdown': count.copy(),
        'coverage': round(coverage, 3),
        'status': status
    })

    print(f"Saniyə {frame_count//30}: {total} avtomobil | Coverage: {coverage:.1%} | {status}")

    frame_count += 1

    # Demo üçün ilk 60 saniyə kifayətdir
    if frame_count > 1800:
        break

cap.release()
print(f"\nCəmi analiz edilən kadr: {len(frame_results)}")

Saniyə 0: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 1: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 2: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 3: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 4: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 5: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 6: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 7: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 8: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 9: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 10: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 11: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 12: 1 avtomobil | Coverage: 1.2% | Boş
Saniyə 13: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 14: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 15: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 16: 2 avtomobil | Coverage: 2.1% | Boş
Saniyə 17: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 18: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 19: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 20: 0 avtomobil | Coverage: 0.0% | Boş
Saniyə 21: 0 avtomobil | Coverage: 0.0% | Bo

In [13]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow
from IPython.display import HTML
from base64 import b64encode

model = YOLO("yolov8n.pt")
VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

video_path = "/content/drive/MyDrive/highway_video.mp4"

cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_path = "/content/drive/MyDrive/traffic_project/detected_output.mp4"
writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

print(f"Processing: {total} frames, {fps} FPS, {w}x{h}")

frame_num = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_num += 1

    results = model(frame, verbose=False)[0]
    frame_area = w * h
    total_box_area = 0
    count = 0

    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        if cls_id in VEHICLE_CLASSES and conf > 0.3:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            total_box_area += (x2-x1) * (y2-y1)
            count += 1
            color = (0,255,0) if cls_id == 2 else (255,165,0)
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, f"{VEHICLE_CLASSES[cls_id]} {conf:.0%}",
                       (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    coverage = total_box_area / frame_area
    if coverage > 0.45: status, scolor = "CONGESTED", (0,0,255)
    elif coverage > 0.25: status, scolor = "HEAVY", (0,100,255)
    elif coverage > 0.10: status, scolor = "NORMAL", (0,200,255)
    else: status, scolor = "FREE FLOW", (0,200,0)

    cv2.rectangle(frame, (10,10), (400,110), (0,0,0), -1)
    cv2.putText(frame, f"Vehicles: {count}", (20,40),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.putText(frame, f"Coverage: {coverage:.1%}", (20,70),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    cv2.putText(frame, f"Status: {status}", (20,100),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, scolor, 2)

    writer.write(frame)

    if frame_num % 100 == 0:
        print(f"  {frame_num}/{total} frames done...")

cap.release()
writer.release()
print(f"Done! Saved: {output_path}")

# Play video in Colab
video_data = open(output_path, "rb").read()
video_b64 = b64encode(video_data).decode()
HTML(f"""
<video width="800" controls autoplay>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
""")

Output hidden; open in https://colab.research.google.com to view.

#Here is diffe

In [1]:
!pip install lightgbm

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split

np.random.seed(42)
n_samples = 50000

# Generate features matching your architecture's 33 features
data = {
    # Real-time detection features
    'current_count': np.random.randint(0, 200, n_samples),
    'coverage_ratio': np.random.uniform(0, 1, n_samples),

    # Lag features
    'count_5min_ago': np.random.randint(0, 200, n_samples),
    'count_15min_ago': np.random.randint(0, 200, n_samples),
    'count_30min_ago': np.random.randint(0, 200, n_samples),
    'count_1h_ago': np.random.randint(0, 200, n_samples),
    'count_yesterday_same_hour': np.random.randint(0, 200, n_samples),
    'avg_last_30min': np.random.uniform(0, 200, n_samples),
    'std_last_30min': np.random.uniform(0, 50, n_samples),
    'trend_30min': np.random.uniform(-50, 50, n_samples),  # increasing or decreasing

    # Weather features
    'temperature': np.random.uniform(-5, 40, n_samples),
    'rain_mm': np.random.exponential(2, n_samples),
    'wind_speed': np.random.uniform(0, 30, n_samples),
    'visibility': np.random.uniform(0.1, 10, n_samples),

    # Calendar features
    'hour': np.random.randint(0, 24, n_samples),
    'minute': np.random.randint(0, 60, n_samples),
    'day_of_week': np.random.randint(0, 7, n_samples),
    'is_weekend': np.random.randint(0, 2, n_samples),
    'is_holiday': np.random.randint(0, 2, n_samples),
    'month': np.random.randint(1, 13, n_samples),

    # Node features
    'node_type': np.random.randint(0, 2, n_samples),  # 0=road, 1=metro
    'node_capacity': np.random.randint(50, 500, n_samples),
    'lane_count': np.random.randint(1, 6, n_samples),

    # Cross features
    'rain_x_peak': np.zeros(n_samples),
    'holiday_x_hour': np.zeros(n_samples),
    'event_nearby': np.random.randint(0, 2, n_samples),
    'event_distance_km': np.random.uniform(0, 10, n_samples),

    # Neighbor features
    'neighbor_avg_count': np.random.uniform(0, 200, n_samples),
    'neighbor_max_count': np.random.randint(0, 300, n_samples),
    'neighbor_trend': np.random.uniform(-50, 50, n_samples),

    # Historical patterns
    'avg_this_hour_weekday': np.random.uniform(0, 200, n_samples),
    'avg_this_hour_weekend': np.random.uniform(0, 150, n_samples),
    'max_this_hour_ever': np.random.randint(50, 400, n_samples),
}

df = pd.DataFrame(data)

# Create cross features
is_peak = ((df['hour'] >= 7) & (df['hour'] <= 9)) | ((df['hour'] >= 17) & (df['hour'] <= 19))
df['rain_x_peak'] = df['rain_mm'] * is_peak.astype(int)
df['holiday_x_hour'] = df['is_holiday'] * df['hour']

# Generate realistic target: congestion % in 1 hour
base = df['current_count'] / df['node_capacity'] * 100
peak_effect = is_peak.astype(int) * 20
rain_effect = np.clip(df['rain_mm'] * 3, 0, 25)
trend_effect = df['trend_30min'] * 0.5
holiday_effect = df['is_holiday'] * (-15)
weekend_effect = df['is_weekend'] * (-10)
noise = np.random.normal(0, 8, n_samples)

df['target_congestion_1h'] = np.clip(
    base + peak_effect + rain_effect + trend_effect + holiday_effect + weekend_effect + noise,
    0, 100
)

print(f"Dataset: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Target mean: {df['target_congestion_1h'].mean():.1f}%")

Dataset: (50000, 34)
Features: 33
Target mean: 43.0%


In [3]:
features = [c for c in df.columns if c != 'target_congestion_1h']
X = df[features]
y = df['target_congestion_1h']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 8,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

model_lgb = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[test_data],
    callbacks=[lgb.log_evaluation(50)]
)

# Evaluate
from sklearn.metrics import mean_absolute_error, r2_score
y_pred = model_lgb.predict(X_test)
print(f"\nMAE: {mean_absolute_error(y_test, y_pred):.2f}%")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': model_lgb.feature_importance()
}).sort_values('importance', ascending=False)
print("\nTop 10 Features:")
print(importance.head(10))

[50]	valid_0's l1: 9.04514
[100]	valid_0's l1: 6.284
[150]	valid_0's l1: 5.85071
[200]	valid_0's l1: 5.76551
[250]	valid_0's l1: 5.74452
[300]	valid_0's l1: 5.72876
[350]	valid_0's l1: 5.72986
[400]	valid_0's l1: 5.73097
[450]	valid_0's l1: 5.70197
[500]	valid_0's l1: 5.6947

MAE: 5.69%
R2 Score: 0.9529

Top 10 Features:
                  feature  importance
0           current_count        1549
21          node_capacity        1434
9             trend_30min        1260
11                rain_mm         883
23            rain_x_peak         797
18             is_holiday         492
12             wind_speed         425
30  avg_this_hour_weekday         415
29         neighbor_trend         411
5            count_1h_ago         409


In [5]:
import joblib
import os
os.makedirs('/content/drive/MyDrive/traffic_project', exist_ok=True)

model_lgb.save_model('/content/drive/MyDrive/traffic_project/lightgbm_congestion.txt')
joblib.dump(features, '/content/drive/MyDrive/traffic_project/lgb_features.pkl')
print("Saved!")

Saved!


#Data fusion layer

In [6]:
import numpy as np
import time
from datetime import datetime

class DataFusion:
    def __init__(self):
        self.node_history = {}  # stores last N readings per node
        self.history_length = 72  # last 6 hours at 5-sec intervals = 72 readings per 1 min

    def update_node(self, source_id, source_type, count, density_pct):
        """Called every 5 seconds when new detection arrives"""
        timestamp = datetime.now().isoformat()

        if source_id not in self.node_history:
            self.node_history[source_id] = {
                'type': source_type,
                'readings': []
            }

        self.node_history[source_id]['readings'].append({
            'timestamp': timestamp,
            'count': count,
            'density_pct': density_pct
        })

        # Keep only last N readings
        self.node_history[source_id]['readings'] = \
            self.node_history[source_id]['readings'][-self.history_length:]

    def get_lag_features(self, source_id):
        """Calculate lag features from history"""
        readings = self.node_history.get(source_id, {}).get('readings', [])

        if len(readings) == 0:
            return {
                'count_5min_ago': 0,
                'count_15min_ago': 0,
                'count_30min_ago': 0,
                'count_1h_ago': 0,
                'avg_last_30min': 0,
                'std_last_30min': 0,
                'trend_30min': 0
            }

        counts = [r['count'] for r in readings]
        current = counts[-1]

        # Approximate lag values based on available readings
        # At 5-sec intervals: 5min=60 readings, 15min=180, 30min=360
        count_5min = counts[-60] if len(counts) >= 60 else counts[0]
        count_15min = counts[-180] if len(counts) >= 180 else counts[0]
        count_30min = counts[-360] if len(counts) >= 360 else counts[0]
        count_1h = counts[-720] if len(counts) >= 720 else counts[0]

        recent = counts[-360:] if len(counts) >= 360 else counts  # last 30 min

        return {
            'count_5min_ago': count_5min,
            'count_15min_ago': count_15min,
            'count_30min_ago': count_30min,
            'count_1h_ago': count_1h,
            'avg_last_30min': np.mean(recent),
            'std_last_30min': np.std(recent),
            'trend_30min': current - count_30min
        }

    def build_lgb_features(self, source_id, node_capacity, lane_count=0):
        """Build feature vector for LightGBM prediction"""
        readings = self.node_history.get(source_id, {}).get('readings', [])
        source_type = self.node_history.get(source_id, {}).get('type', 'road')

        if len(readings) == 0:
            return None

        current = readings[-1]
        lags = self.get_lag_features(source_id)
        now = datetime.now()

        # Get neighbor data
        neighbor_counts = []
        for nid, ndata in self.node_history.items():
            if nid != source_id and ndata['readings']:
                neighbor_counts.append(ndata['readings'][-1]['count'])

        features = {
            'current_count': current['count'],
            'coverage_ratio': current['density_pct'] / 100,

            # Lag features
            'count_5min_ago': lags['count_5min_ago'],
            'count_15min_ago': lags['count_15min_ago'],
            'count_30min_ago': lags['count_30min_ago'],
            'count_1h_ago': lags['count_1h_ago'],
            'count_yesterday_same_hour': lags['count_1h_ago'],  # placeholder
            'avg_last_30min': lags['avg_last_30min'],
            'std_last_30min': lags['std_last_30min'],
            'trend_30min': lags['trend_30min'],

            # No weather — set neutral values
            'temperature': 20,
            'rain_mm': 0,
            'wind_speed': 0,
            'visibility': 10,

            # Calendar from system clock
            'hour': now.hour,
            'minute': now.minute,
            'day_of_week': now.weekday(),
            'is_weekend': 1 if now.weekday() >= 5 else 0,
            'is_holiday': 0,
            'month': now.month,

            # Node info
            'node_type': 1 if source_type == 'metro' else 0,
            'node_capacity': node_capacity,
            'lane_count': lane_count,

            # Cross features
            'rain_x_peak': 0,
            'holiday_x_hour': 0,
            'event_nearby': 0,
            'event_distance_km': 10,

            # Neighbor features
            'neighbor_avg_count': np.mean(neighbor_counts) if neighbor_counts else 0,
            'neighbor_max_count': max(neighbor_counts) if neighbor_counts else 0,
            'neighbor_trend': 0,

            # Historical (placeholder)
            'avg_this_hour_weekday': lags['avg_last_30min'],
            'avg_this_hour_weekend': lags['avg_last_30min'] * 0.7,
            'max_this_hour_ever': current['count'] * 2,
        }

        return features

    def build_convlstm_grid(self, grid_size=32):
        """Build 32x32 grid for ConvLSTM from all node data"""
        grid = np.zeros((grid_size, grid_size))

        # Map each node to a grid cell based on its position
        # In production: use actual GPS coordinates
        # For demo: distribute nodes across grid
        for i, (source_id, data) in enumerate(self.node_history.items()):
            if data['readings']:
                count = data['readings'][-1]['count']
                # Assign to grid cell (demo: spread across grid)
                row = (i * 7) % grid_size
                col = (i * 13) % grid_size
                grid[row, col] = count

        return grid

    def fuse_predictions(self, source_id, lgb_prediction, convlstm_prediction):
        """
        Combine LightGBM and ConvLSTM predictions
        - Nodes with camera: trust LightGBM more (0.6)
        - Nodes without camera: trust ConvLSTM more (0.4 -> 1.0)
        """
        has_camera = source_id in self.node_history

        if has_camera:
            final = 0.6 * lgb_prediction + 0.4 * convlstm_prediction
        else:
            final = convlstm_prediction

        return round(final, 1)

    def get_fused_state(self, source_id, lgb_model, convlstm_grid_prediction, node_capacity, lane_count=0):
        """
        Full fusion: get features, predict, combine
        Returns unified node state
        """
        # LightGBM prediction
        features = self.build_lgb_features(source_id, node_capacity, lane_count)
        if features:
            import pandas as pd
            df_input = pd.DataFrame([features])
            lgb_pred = lgb_model.predict(df_input)[0]
        else:
            lgb_pred = 50  # default

        # ConvLSTM prediction for this node's grid cell
        # In production: map source_id to grid cell and read value
        convlstm_pred = convlstm_grid_prediction  # passed from ConvLSTM output

        # Fuse
        final_prediction = self.fuse_predictions(source_id, lgb_pred, convlstm_pred)

        # Current state
        readings = self.node_history.get(source_id, {}).get('readings', [])
        current_pct = readings[-1]['density_pct'] if readings else 0

        # Trend
        lags = self.get_lag_features(source_id)
        trend_val = lags['trend_30min']
        if trend_val > 10:
            trend = "artır"
        elif trend_val < -10:
            trend = "azalır"
        else:
            trend = "sabit"

        # Status
        if final_prediction < 25:
            status = "boş olacaq"
        elif final_prediction < 50:
            status = "normal olacaq"
        elif final_prediction < 75:
            status = "sıxlaşır"
        else:
            status = "tıxac olacaq"

        return {
            'source_id': source_id,
            'current': current_pct,
            'forecast_1h': final_prediction,
            'trend': trend,
            'status': status,
            'lgb_raw': round(lgb_pred, 1),
            'convlstm_raw': round(convlstm_pred, 1)
        }


# ============ TEST IT ============

fusion = DataFusion()

# Simulate YOLO road camera data (feeds ConvLSTM)
for i in range(10):
    fusion.update_node('c1', 'road', count=50 + i*3, density_pct=40 + i*2)
    fusion.update_node('c2', 'road', count=80 + i*2, density_pct=60 + i)

# Simulate CSRNet metro data
for i in range(10):
    fusion.update_node('m1', 'metro', count=120 + i*5, density_pct=55 + i*3)
    fusion.update_node('m2', 'metro', count=90 + i*2, density_pct=45 + i)

# Build features for LightGBM
features = fusion.build_lgb_features('c1', node_capacity=200, lane_count=3)
print("LightGBM features for c1:")
print(f"  current_count: {features['current_count']}")
print(f"  trend_30min: {features['trend_30min']}")
print(f"  coverage_ratio: {features['coverage_ratio']}")

# Build grid for ConvLSTM
grid = fusion.build_convlstm_grid()
print(f"\nConvLSTM grid shape: {grid.shape}")
print(f"Non-zero cells: {np.count_nonzero(grid)}")

# Fuse predictions (using your trained LightGBM)
# For demo without model loaded:
print("\nFusion test (mock predictions):")
lgb_pred = 72.5
convlstm_pred = 68.0
final = fusion.fuse_predictions('c1', lgb_pred, convlstm_pred)
print(f"  LightGBM: {lgb_pred}%, ConvLSTM: {convlstm_pred}%")
print(f"  Fused: {final}% (0.6*LGB + 0.4*ConvLSTM)")

# Full state with real LightGBM model
# Uncomment when model is loaded:
# state = fusion.get_fused_state('c1', model_lgb, convlstm_pred=68.0, node_capacity=200, lane_count=3)
# print(f"\nFull state: {state}")

print("\nData Fusion ready!")

LightGBM features for c1:
  current_count: 77
  trend_30min: 27
  coverage_ratio: 0.58

ConvLSTM grid shape: (32, 32)
Non-zero cells: 4

Fusion test (mock predictions):
  LightGBM: 72.5%, ConvLSTM: 68.0%
  Fused: 70.7% (0.6*LGB + 0.4*ConvLSTM)

Data Fusion ready!


In [7]:
import numpy as np
from datetime import datetime

class DecisionEngine:
    def __init__(self, grid_size=32):
        self.grid_size = grid_size
        self.node_states = {}
        self.congestion_zones = []

    def update_node_state(self, fused_state):
        """Store fused state for each node"""
        self.node_states[fused_state['source_id']] = fused_state

    def detect_anomaly(self, source_id, current, historical_avg):
        """Flag if current value deviates sharply from normal"""
        if historical_avg == 0:
            return False
        deviation = abs(current - historical_avg) / historical_avg
        return deviation > 0.5  # 50% deviation = anomaly

    def find_congestion_zones(self, convlstm_grid, threshold=0.6):
        """
        Find congestion zones from ConvLSTM grid output
        1. Find high-value cells
        2. Cluster nearby cells
        3. Calculate zone properties
        """
        self.congestion_zones = []

        # Normalize grid to 0-1
        if convlstm_grid.max() > 0:
            normalized = convlstm_grid / convlstm_grid.max()
        else:
            return []

        # Find cells above threshold
        hot_cells = np.argwhere(normalized > threshold)

        if len(hot_cells) == 0:
            return []

        # Simple clustering: group nearby cells
        clusters = []
        used = set()

        for i, cell in enumerate(hot_cells):
            if i in used:
                continue

            cluster = [cell]
            used.add(i)

            # Find all neighbors
            for j, other in enumerate(hot_cells):
                if j in used:
                    continue
                # If within 3 cells distance, same cluster
                if abs(cell[0] - other[0]) <= 3 and abs(cell[1] - other[1]) <= 3:
                    cluster.append(other)
                    used.add(j)

            clusters.append(cluster)

        # Build zone info for each cluster
        for cluster in clusters:
            cells = np.array(cluster)
            center_row = cells[:, 0].mean()
            center_col = cells[:, 1].mean()

            # Zone size based on number of cells
            radius = len(cells) * 0.5  # km approximation

            # Severity based on average grid value
            avg_value = np.mean([convlstm_grid[c[0], c[1]] for c in cells])
            max_value = convlstm_grid.max()
            severity_pct = (avg_value / max_value * 100) if max_value > 0 else 0

            if severity_pct > 80:
                severity = "çox yüksək"
            elif severity_pct > 60:
                severity = "yüksək"
            elif severity_pct > 40:
                severity = "orta"
            else:
                severity = "aşağı"

            # Spreading direction: compare with neighboring cells
            spread_row = 0
            spread_col = 0
            for c in cells:
                r, col = c[0], c[1]
                if r + 1 < self.grid_size:
                    spread_row += convlstm_grid[r + 1, col]
                if r - 1 >= 0:
                    spread_row -= convlstm_grid[r - 1, col]
                if col + 1 < self.grid_size:
                    spread_col += convlstm_grid[r, col + 1]
                if col - 1 >= 0:
                    spread_col -= convlstm_grid[r, col - 1]

            if abs(spread_row) > abs(spread_col):
                direction = "cənub" if spread_row > 0 else "şimal"
            else:
                direction = "şərq" if spread_col > 0 else "qərb"

            zone = {
                'center_grid': [round(center_row, 1), round(center_col, 1)],
                'radius_km': round(radius, 1),
                'severity': severity,
                'severity_pct': round(severity_pct, 1),
                'spreading_towards': direction,
                'cell_count': len(cells),
                'starts_at': datetime.now().strftime('%H:%M'),
                'duration_min': int(30 + severity_pct * 0.5)  # estimate
            }

            self.congestion_zones.append(zone)

        return self.congestion_zones

    def process_all_nodes(self, fusion, lgb_model, convlstm_grid_pred, node_configs):
        """
        Run decision engine for all nodes
        node_configs: dict of {source_id: {capacity, lane_count}}
        """
        # Update states for all nodes
        for source_id, config in node_configs.items():
            state = fusion.get_fused_state(
                source_id,
                lgb_model,
                convlstm_grid_prediction=convlstm_grid_pred.get(source_id, 50),
                node_capacity=config['capacity'],
                lane_count=config.get('lane_count', 0)
            )

            # Check anomaly
            state['anomaly'] = self.detect_anomaly(
                source_id,
                state['current'],
                config.get('historical_avg', state['current'])
            )

            self.update_node_state(state)

        # Detect congestion zones from ConvLSTM grid
        # In production: use actual ConvLSTM output grid
        # For demo: build from node data
        grid = fusion.build_convlstm_grid()
        zones = self.find_congestion_zones(grid)

        return self.get_full_output()

    def get_full_output(self):
        """Return complete decision output matching your architecture spec"""
        return {
            'timestamp': datetime.now().isoformat(),
            'congestion_zones': self.congestion_zones,
            'node_states': self.node_states,
            'summary': {
                'total_nodes': len(self.node_states),
                'congested_nodes': sum(
                    1 for s in self.node_states.values()
                    if s['forecast_1h'] > 75
                ),
                'zones_detected': len(self.congestion_zones)
            }
        }


# ============ TEST IT ============

engine = DecisionEngine(grid_size=32)

# Use the fusion object from before
# Simulate ConvLSTM predictions per node
convlstm_preds = {
    'c1': 70.0,
    'c2': 55.0,
    'm1': 80.0,
    'm2': 45.0
}

# Node configurations
node_configs = {
    'c1': {'capacity': 200, 'lane_count': 3, 'historical_avg': 60},
    'c2': {'capacity': 150, 'lane_count': 2, 'historical_avg': 70},
    'm1': {'capacity': 300, 'lane_count': 0, 'historical_avg': 100},
    'm2': {'capacity': 250, 'lane_count': 0, 'historical_avg': 80}
}

# Run decision engine (using mock LGB for now)
# For real: pass model_lgb
class MockLGB:
    def predict(self, df):
        return [65.0]

output = engine.process_all_nodes(fusion, MockLGB(), convlstm_preds, node_configs)

# Print results
import json
print("=== DECISION ENGINE OUTPUT ===\n")
print(f"Timestamp: {output['timestamp']}")
print(f"\nSummary:")
print(f"  Total nodes: {output['summary']['total_nodes']}")
print(f"  Congested: {output['summary']['congested_nodes']}")
print(f"  Zones: {output['summary']['zones_detected']}")

print(f"\nNode States:")
for nid, state in output['node_states'].items():
    print(f"  {nid}: current={state['current']}%, forecast={state['forecast_1h']}%, "
          f"trend={state['trend']}, status={state['status']}")

print(f"\nCongestion Zones:")
for i, zone in enumerate(output['congestion_zones']):
    print(f"  Zone {i+1}: severity={zone['severity']}, "
          f"radius={zone['radius_km']}km, "
          f"spreading={zone['spreading_towards']}, "
          f"duration={zone['duration_min']}min")

print("\nDecision Engine ready!")

=== DECISION ENGINE OUTPUT ===

Timestamp: 2026-04-08T23:28:36.984807

Summary:
  Total nodes: 4
  Congested: 0
  Zones: 2

Node States:
  c1: current=58%, forecast=67.0%, trend=artır, status=sıxlaşır
  c2: current=69%, forecast=61.0%, trend=artır, status=sıxlaşır
  m1: current=82%, forecast=71.0%, trend=artır, status=sıxlaşır
  m2: current=54%, forecast=57.0%, trend=artır, status=sıxlaşır

Congestion Zones:
  Zone 1: severity=çox yüksək, radius=0.5km, spreading=qərb, duration=80min
  Zone 2: severity=yüksək, radius=0.5km, spreading=qərb, duration=62min

Decision Engine ready!


In [8]:
import heapq
import numpy as np
from datetime import datetime, timedelta

class RoutingEngine:
    def __init__(self, decision_engine):
        self.decision = decision_engine
        self.graph = {}  # adjacency list
        self.node_positions = {}  # node_id -> (lat, lon)
        self.node_modes = {}  # node_id -> transport mode

    def add_node(self, node_id, lat, lon, mode='road'):
        """Add a node to the routing graph"""
        self.node_positions[node_id] = (lat, lon)
        self.node_modes[node_id] = mode
        if node_id not in self.graph:
            self.graph[node_id] = []

    def add_edge(self, from_id, to_id, base_time_min, distance_km, mode='road'):
        """Add edge with base travel time"""
        self.graph.setdefault(from_id, [])
        self.graph[from_id].append({
            'to': to_id,
            'base_time': base_time_min,
            'distance': distance_km,
            'mode': mode
        })
        # Bidirectional
        self.graph.setdefault(to_id, [])
        self.graph[to_id].append({
            'to': from_id,
            'base_time': base_time_min,
            'distance': distance_km,
            'mode': mode
        })

    def get_dynamic_cost(self, node_id, edge, arrival_time_min):
        """
        Calculate dynamic edge cost based on:
        - Decision engine forecasts
        - Time of arrival at that segment
        """
        base = edge['base_time']
        mode = edge['mode']

        # Get node forecast from decision engine
        state = self.decision.node_states.get(node_id, None)

        if state is None:
            return base

        forecast = state['forecast_1h']
        trend = state['trend']

        # Congestion multiplier
        if forecast > 85:
            multiplier = 3.0  # tıxac — 3x slower
        elif forecast > 70:
            multiplier = 2.0  # sıx — 2x slower
        elif forecast > 50:
            multiplier = 1.5  # normal-sıx
        else:
            multiplier = 1.0  # normal

        # Trend adjustment — if getting worse, add extra cost
        if trend == 'artır':
            multiplier += 0.3
        elif trend == 'azalır':
            multiplier -= 0.2

        # Check if node is in a congestion zone
        for zone in self.decision.congestion_zones:
            if zone['severity'] in ['yüksək', 'çox yüksək']:
                # If this node is near zone center, extra penalty
                multiplier += 0.5
                break

        # Mode-specific adjustments
        if mode == 'metro':
            # Metro less affected by road congestion
            multiplier = 1.0 + (multiplier - 1.0) * 0.3
        elif mode == 'walk':
            # Walking not affected by traffic
            multiplier = 1.0

        return base * max(multiplier, 1.0)

    def heuristic(self, node_a, node_b):
        """A* heuristic: straight-line distance in minutes"""
        if node_a not in self.node_positions or node_b not in self.node_positions:
            return 0
        lat1, lon1 = self.node_positions[node_a]
        lat2, lon2 = self.node_positions[node_b]
        # Approximate distance in km
        dist = ((lat1-lat2)**2 + (lon1-lon2)**2) ** 0.5 * 111
        # Assume 30 km/h average
        return dist / 30 * 60

    def find_route(self, start, end, departure_time=None):
        """
        Time-dependent A* routing
        Looks at forecast for ARRIVAL time at each segment, not departure time
        """
        if departure_time is None:
            departure_time = 0  # minutes from now

        # Priority queue: (total_cost, current_time, node, path)
        queue = [(0, departure_time, start, [start], [])]
        visited = {}

        while queue:
            cost, current_time, node, path, edges = heapq.heappop(queue)

            if node == end:
                return {
                    'path': path,
                    'edges': edges,
                    'total_time': round(cost, 1),
                    'departure': departure_time,
                    'arrival': round(departure_time + cost, 1)
                }

            if node in visited and visited[node] <= cost:
                continue
            visited[node] = cost

            for edge in self.graph.get(node, []):
                next_node = edge['to']

                # KEY: use arrival time at next segment for forecast
                dynamic_cost = self.get_dynamic_cost(
                    next_node, edge, current_time + edge['base_time']
                )

                new_cost = cost + dynamic_cost
                new_time = current_time + dynamic_cost
                h = self.heuristic(next_node, end)

                heapq.heappush(queue, (
                    new_cost + h,
                    new_time,
                    next_node,
                    path + [next_node],
                    edges + [{
                        'from': node,
                        'to': next_node,
                        'mode': edge['mode'],
                        'time': round(dynamic_cost, 1),
                        'distance': edge['distance']
                    }]
                ))

        return None  # No route found

    def find_multimodal_routes(self, start, end):
        """
        Find multiple route options with different transport modes
        Returns ranked alternatives
        """
        routes = []

        # Route 1: Direct road route
        road_route = self.find_route(start, end)
        if road_route:
            road_route['type'] = 'road'
            road_route['label'] = 'Avtomobil'
            road_route['risk'] = self.calculate_risk(road_route)
            routes.append(road_route)

        # Route 2: Find metro-assisted route if metro nodes exist
        metro_nodes = [n for n, m in self.node_modes.items() if m == 'metro']

        if metro_nodes:
            # Find nearest metro to start and end
            best_metro_route = None
            best_time = float('inf')

            for m_start in metro_nodes:
                for m_end in metro_nodes:
                    if m_start == m_end:
                        continue

                    # Walk to metro -> metro ride -> walk to destination
                    leg1 = self.find_route(start, m_start)
                    leg2 = self.find_route(m_start, m_end)
                    leg3 = self.find_route(m_end, end)

                    if leg1 and leg2 and leg3:
                        total = leg1['total_time'] + leg2['total_time'] + leg3['total_time']
                        if total < best_time:
                            best_time = total
                            best_metro_route = {
                                'path': leg1['path'] + leg2['path'][1:] + leg3['path'][1:],
                                'edges': leg1['edges'] + leg2['edges'] + leg3['edges'],
                                'total_time': round(total, 1),
                                'legs': [
                                    {'mode': 'walk', 'time': leg1['total_time']},
                                    {'mode': 'metro', 'time': leg2['total_time']},
                                    {'mode': 'walk', 'time': leg3['total_time']}
                                ],
                                'type': 'multimodal',
                                'label': 'Metro + Piyada'
                            }

            if best_metro_route:
                best_metro_route['risk'] = self.calculate_risk(best_metro_route)
                routes.append(best_metro_route)

        # Sort by total time
        routes.sort(key=lambda r: r['total_time'])

        # Add reliability score
        for i, route in enumerate(routes):
            route['rank'] = i + 1
            route['reliability'] = max(100 - route['risk'] * 10, 20)

        return routes

    def calculate_risk(self, route):
        """Calculate congestion risk for a route (0-10)"""
        risk = 0
        for edge in route.get('edges', []):
            node = edge['to']
            state = self.decision.node_states.get(node, {})
            forecast = state.get('forecast_1h', 50)
            if forecast > 80:
                risk += 3
            elif forecast > 60:
                risk += 2
            elif forecast > 40:
                risk += 1

        return min(risk, 10)

    def format_route(self, route):
        """Pretty print a route"""
        if not route:
            return "Marşrut tapılmadı"

        lines = []
        lines.append(f"📍 Marşrut: {route.get('label', 'Standard')}")
        lines.append(f"⏱  Ümumi vaxt: {route['total_time']} dəqiqə")
        lines.append(f"📊 Etibarlılıq: {route.get('reliability', 'N/A')}%")
        lines.append(f"⚠️  Risk: {route.get('risk', 'N/A')}/10")
        lines.append("")

        if 'legs' in route:
            for leg in route['legs']:
                mode_icon = '🚇' if leg['mode'] == 'metro' else '🚶'
                lines.append(f"  {mode_icon} {leg['mode']}: {leg['time']} dəq")
        else:
            for edge in route['edges']:
                mode_icon = {'road': '🚗', 'metro': '🚇', 'walk': '🚶'}.get(edge['mode'], '➡️')
                lines.append(f"  {mode_icon} {edge['from']} → {edge['to']}: {edge['time']} dəq ({edge['mode']})")

        return '\n'.join(lines)


# ============ BUILD DEMO NETWORK ============

router = RoutingEngine(engine)

# Add road nodes (Baku example)
router.add_node('A', 40.4093, 49.8671, 'road')   # Start - Yasamal
router.add_node('c1', 40.4050, 49.8700, 'road')   # Camera 1 - Neftçilər
router.add_node('c2', 40.3980, 49.8750, 'road')   # Camera 2 - Babək pr
router.add_node('B', 40.3920, 49.8800, 'road')     # End - Həzi Aslanov

# Add metro nodes
router.add_node('m1', 40.4080, 49.8690, 'metro')  # Gənclik metro
router.add_node('m2', 40.3950, 49.8770, 'metro')  # 28 May metro

# Add road edges
router.add_edge('A', 'c1', base_time_min=5, distance_km=1.2, mode='road')
router.add_edge('c1', 'c2', base_time_min=8, distance_km=2.0, mode='road')
router.add_edge('c2', 'B', base_time_min=6, distance_km=1.5, mode='road')
router.add_edge('A', 'c2', base_time_min=12, distance_km=3.0, mode='road')  # alternative road

# Add walk edges to metro
router.add_edge('A', 'm1', base_time_min=4, distance_km=0.3, mode='walk')
router.add_edge('m2', 'B', base_time_min=5, distance_km=0.4, mode='walk')

# Add metro edge
router.add_edge('m1', 'm2', base_time_min=6, distance_km=3.5, mode='metro')

# Walk connections
router.add_edge('c1', 'm1', base_time_min=3, distance_km=0.2, mode='walk')
router.add_edge('c2', 'm2', base_time_min=3, distance_km=0.2, mode='walk')

# ============ FIND ROUTES ============

print("=" * 50)
print("ROUTING: A (Yasamal) → B (Həzi Aslanov)")
print("=" * 50)

routes = router.find_multimodal_routes('A', 'B')

for route in routes:
    print(f"\n{'='*40}")
    print(router.format_route(route))

print(f"\n{'='*50}")
print(f"Təklif: {routes[0]['label']} ({routes[0]['total_time']} dəq)")
print(f"{'='*50}")


ROUTING: A (Yasamal) → B (Həzi Aslanov)

📍 Marşrut: Metro + Piyada
⏱  Ümumi vaxt: 17.3 dəqiqə
📊 Etibarlılıq: 60%
⚠️  Risk: 4/10

  🚶 walk: 4.0 dəq
  🚇 metro: 8.3 dəq
  🚶 walk: 5.0 dəq

📍 Marşrut: Avtomobil
⏱  Ümumi vaxt: 22.6 dəqiqə
📊 Etibarlılıq: 60%
⚠️  Risk: 4/10

  🚶 A → m1: 4.0 dəq (walk)
  🚇 m1 → m2: 8.3 dəq (metro)
  🚶 m2 → B: 5 dəq (walk)

Təklif: Metro + Piyada (17.3 dəq)
